# Clinical Analytics Chatbot
**Drug R&D Assistant — Dataiku Notebook UI**

Run all cells in order. The interactive chat UI launches in the last cell (Cell 4).

**Before running:** set your LLM Mesh connection ID in Cell 2.

`lib/python/` is automatically on the Python path in all Dataiku notebooks — no path manipulation needed.

In [ ]:
# Cell 1 — Initialise Panel
import panel as pn
pn.extension('bokeh', sizing_mode='stretch_width')

In [ ]:
# Cell 2 — Configuration
# Replace with your actual Dataiku LLM Mesh connection ID
LLM_CONNECTION_ID = "YOUR_LLM_CONNECTION_ID"

In [ ]:
# Cell 3 — Initialise backend
import sys

# Flush cached backend modules so updated code is always loaded fresh
for _mod in list(sys.modules.keys()):
    if _mod.startswith('backend') or _mod.startswith('frontend'):
        del sys.modules[_mod]

import yaml
from pathlib import Path
import backend

config_dir = Path(backend.__file__).parent.parent / "config"
with open(config_dir / "llm_config.yaml") as f:
    config = yaml.safe_load(f)
config["llm_mesh"]["connection_id"] = LLM_CONNECTION_ID

from backend.state.session_store import SessionStore
from backend.orchestrator.orchestrator import Orchestrator

session_store = SessionStore(timeout_minutes=60)

# Expose Dataiku project variables as env vars so WebSearchClient can find serp_api_key
try:
    import dataiku as _dku
    _proj_vars = _dku.get_custom_variables()
    if "serp_api_key" in _proj_vars:
        import os as _os2
        _os2.environ.setdefault("serp_api_key", _proj_vars["serp_api_key"])
        print("serp_api_key loaded from Dataiku project variables")
except Exception:
    pass  # not running inside Dataiku, or variable not set

orchestrator = Orchestrator(session_store, config=config)

# Apply all notebook-specific patches (call logging, site matching agent,
# JSON repair, benchmarking prompt, confidence thresholds, general-knowledge fallback)
from backend.notebook_patches import apply_patches
apply_patches(orchestrator)

# LLM connectivity test
try:
    _test = orchestrator.llm.complete(
        [{"role": "user", "content": "Reply with the single word: OK"}],
    )
    print(f"LLM connectivity test PASSED — response: {_test[:80]!r}")
except Exception as _e:
    print(f"LLM connectivity test FAILED: {_e}")


In [ ]:
# Cell 4 — Build and launch the interactive UI
from frontend.panel_app import build_app

app = build_app(orchestrator, session_store)
app.servable()
app
